In [6]:
# ============================================================
# ASSIGNMENT SIMILARITY / PLAGIARISM DETECTOR
# NLP PROJECT USING TF-IDF AND COSINE SIMILARITY
# ============================================================

# -----------------------------
# 1. IMPORT LIBRARIES
# -----------------------------

import pandas as pd
import numpy as np
import re
import nltk
import matplotlib.pyplot as plt
import seaborn as sns

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error


# -----------------------------
# 2. DOWNLOAD NLTK DATA
# -----------------------------

nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")


# -----------------------------
# 3. LOAD DATASET
# -----------------------------

df = pd.read_csv("textual_similarity.csv")

print("\n========== DATASET ==========")

print("\nFirst 5 records:")
print(df.head())

print("\nDataset shape:")
print(df.shape)

print("\nColumn names:")
print(df.columns)


# -----------------------------
# 4. SELECT TEXT COLUMNS
# -----------------------------

# The Kaggle dataset contains text1 and text2

df = df[["text1", "text2"]]


# -----------------------------
# 5. REMOVE MISSING VALUES
# -----------------------------

print("\nMissing values before cleaning:")
print(df.isnull().sum())

df = df.dropna()

print("\nDataset shape after removing missing values:")
print(df.shape)


# -----------------------------
# 6. REMOVE DUPLICATES
# -----------------------------

print("\nDuplicate rows:")
print(df.duplicated().sum())

df = df.drop_duplicates()

print("\nDataset shape after removing duplicates:")
print(df.shape)


# ============================================================
# 7. TEXT PREPROCESSING
# ============================================================

stop_words = set(stopwords.words("english"))

lemmatizer = WordNetLemmatizer()


def preprocess_text(text):

    # Convert to string
    text = str(text)

    # Convert to lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)

    # Remove HTML tags
    text = re.sub(r"<.*?>", "", text)

    # Remove numbers
    text = re.sub(r"\d+", "", text)

    # Remove punctuation and special characters
    text = re.sub(r"[^a-zA-Z\s]", " ", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    # Tokenization
    words = text.split()

    # Remove stopwords
    words = [
        word for word in words
        if word not in stop_words
    ]

    # Lemmatization
    words = [
        lemmatizer.lemmatize(word)
        for word in words
    ]

    # Join words
    return " ".join(words)


# -----------------------------
# 8. APPLY PREPROCESSING
# -----------------------------

print("\nPreprocessing text...")

df["clean_text1"] = df["text1"].apply(preprocess_text)

df["clean_text2"] = df["text2"].apply(preprocess_text)


print("\nOriginal text:")
print(df["text1"].iloc[0])

print("\nCleaned text:")
print(df["clean_text1"].iloc[0])


# ============================================================
# 9. COMBINE BOTH TEXT COLUMNS
# ============================================================

all_text = pd.concat(
    [
        df["clean_text1"],
        df["clean_text2"]
    ]
)


# ============================================================
# 10. TF-IDF VECTORIZATION
# ============================================================

print("\nCreating TF-IDF vectors...")

vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2)
)


tfidf_matrix = vectorizer.fit_transform(all_text)


print("\nTF-IDF matrix shape:")
print(tfidf_matrix.shape)


# ============================================================
# 11. SPLIT TF-IDF MATRIX
# ============================================================

number_of_rows = len(df)

tfidf_text1 = tfidf_matrix[:number_of_rows]

tfidf_text2 = tfidf_matrix[number_of_rows:]


# ============================================================
# 12. CALCULATE COSINE SIMILARITY
# ============================================================

similarity_scores = []

print("\nCalculating similarity...")

for i in range(number_of_rows):

    vector1 = tfidf_text1[i]

    vector2 = tfidf_text2[i]

    similarity = cosine_similarity(
        vector1,
        vector2
    )[0][0]

    similarity_scores.append(similarity)


# Add similarity to dataframe

df["similarity"] = similarity_scores


# Convert similarity to percentage

df["similarity_percentage"] = (
    df["similarity"] * 100
)


# ============================================================
# 13. CLASSIFY SIMILARITY
# ============================================================

def classify_similarity(score):

    if score >= 0.80:
        return "Highly Similar / Possible Plagiarism"

    elif score >= 0.60:
        return "Moderately Similar"

    elif score >= 0.40:
        return "Somewhat Similar"

    else:
        return "Low Similarity"


df["result"] = df["similarity"].apply(
    classify_similarity
)


# ============================================================
# 14. DISPLAY RESULTS
# ============================================================

print("\n========== SIMILARITY RESULTS ==========")

print(
    df[
        [
            "text1",
            "text2",
            "similarity_percentage",
            "result"
        ]
    ].head(10)
)


# ============================================================
# 15. DISPLAY TOP SIMILAR DOCUMENTS
# ============================================================

print("\n========== TOP SIMILAR ASSIGNMENTS ==========")

top_similar = df.sort_values(
    by="similarity",
    ascending=False
).head(10)


for index, row in top_similar.iterrows():

    print("\n-----------------------------------")

    print(
        "Similarity:",
        round(row["similarity_percentage"], 2),
        "%"
    )

    print(
        "Result:",
        row["result"]
    )

    print(
        "\nAssignment 1:",
        row["text1"][:300]
    )

    print(
        "\nAssignment 2:",
        row["text2"][:300]
    )


# ============================================================
# 16. DATASET STATISTICS
# ============================================================

print("\n========== STATISTICS ==========")

print(
    "Average similarity:",
    round(
        df["similarity_percentage"].mean(),
        2
    ),
    "%"
)

print(
    "Maximum similarity:",
    round(
        df["similarity_percentage"].max(),
        2
    ),
    "%"
)

print(
    "Minimum similarity:",
    round(
        df["similarity_percentage"].min(),
        2
    ),
    "%"
)


# ============================================================
# 17. RESULT COUNT
# ============================================================

print("\n========== RESULT COUNTS ==========")

print(
    df["result"].value_counts()
)


# ============================================================
# 18. VISUALIZATION - SIMILARITY DISTRIBUTION
# ============================================================

plt.figure(figsize=(10, 6))

plt.hist(
    df["similarity_percentage"],
    bins=20
)

plt.xlabel("Similarity Percentage")

plt.ylabel("Number of Assignment Pairs")

plt.title(
    "Assignment Similarity Distribution"
)

plt.show()


# ============================================================
# 19. VISUALIZATION - RESULT COUNTS
# ============================================================

plt.figure(figsize=(10, 6))

df["result"].value_counts().plot(
    kind="bar"
)

plt.xlabel("Similarity Category")

plt.ylabel("Number of Assignment Pairs")

plt.title(
    "Plagiarism / Similarity Categories"
)

plt.xticks(
    rotation=30,
    ha="right"
)

plt.tight_layout()

plt.show()


# ============================================================
# 20. SAVE RESULTS
# ============================================================

df.to_csv(
    "plagiarism_detection_results.csv",
    index=False
)

print(
    "\nResults saved as "
    "'plagiarism_detection_results.csv'"
)


# ============================================================
# 21. FUNCTION TO CHECK TWO NEW ASSIGNMENTS
# ============================================================

def check_plagiarism(assignment1, assignment2):

    # Preprocess both assignments

    clean1 = preprocess_text(assignment1)

    clean2 = preprocess_text(assignment2)

    # Convert both texts into TF-IDF vectors

    vectors = vectorizer.transform(
        [clean1, clean2]
    )

    # Calculate cosine similarity

    score = cosine_similarity(
        vectors[0],
        vectors[1]
    )[0][0]

    percentage = score * 100

    # Classification

    result = classify_similarity(score)

    print("\n====================================")
    print("      PLAGIARISM CHECK RESULT")
    print("====================================")

    print(
        "\nSimilarity Score:",
        round(score, 4)
    )

    print(
        "Similarity Percentage:",
        round(percentage, 2),
        "%"
    )

    print(
        "Result:",
        result
    )

    print("====================================")

    return score


# ============================================================
# 22. TEST WITH TWO NEW ASSIGNMENTS
# ============================================================

assignment1 = """
Artificial intelligence is a branch of computer science
that allows machines to perform tasks that normally require
human intelligence. Machine learning is an important part
of artificial intelligence.
"""


assignment2 = """
Artificial intelligence is a field of computer science
where machines are designed to perform tasks that usually
need human intelligence. Machine learning is one of the
important areas of artificial intelligence.
"""


check_plagiarism(
    assignment1,
    assignment2
)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\acer\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\acer\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\acer\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


FileNotFoundError: [Errno 2] No such file or directory: 'textual_similarity.csv'